In [ ]:

import pandas 
import geopandas
import os
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.impute import SimpleImputer
from config import *
from misc_utilities import *

In [ ]:

def df_to_las(df,header,out_file_path):
    total_points = len(df)
    record = laspy.ScaleAwarePointRecord.zeros(total_points, header=header)
    out_header = laspy.LasHeader(point_format=header.point_format,version="1.4")
    out_header.x_scale = header.x_scale
    out_header.y_scale = header.y_scale
    out_header.z_scale = header.z_scale
    out_header.offsets = header.offsets            
    out_crs = header.parse_crs()
    out_header.add_crs(out_crs)
    for var1 in df.columns:
        record[var1] = df[var1].values
    with laspy.open(out_file_path, mode='w', header=out_header) as writer:
                writer.write_points(record)




selected_cols = ['X', 'Y', 'Z', 'red', 'green', 'blue','classification']
data_info_df = pandas.read_csv(data_info_file)
filename_paths = {os.path.split(x)[-1]:x for x in data_info_df['filename']}
for las_file in las_vect_dict:
    vect_file = las_vect_dict[las_file]
    shapefile_path = os.path.join(vector_dir,vect_file)
    las_file_path = filename_paths[las_file]
    las_file_name = os.path.splitext(las_file)[0]
    dtm_file = las_file_name+"_dtm.las"
    dtm_file_path = os.path.join(dtm_dir,dtm_file)
    gdf = geopandas.read_file(shapefile_path)
    for row1 in gdf.iterrows():
        id = row1[1]['id']
        geometry = row1[1].geometry
        dtm_record,dtm_header = subset_with_geom(dtm_file_path,geometry)
        dsm_record,dsm_header = subset_with_geom(las_file_path,geometry)
        df = pandas.DataFrame(dtm_record.array)
        filled_df = knn_fill(df)

        dtm_debug_file = os.path.join(debug_dir,f"dtm_debug_{las_file_name}_id_{id}.las")
        dsm_debug_file = os.path.join(debug_dir,f"dsm_debug_{las_file_name}_id_{id}.las")
        terrain_debug_file = os.path.join(debug_dir,f"terrain_debug_{las_file_name}_id_{id}.las")
        df_to_las(filled_df,dtm_header,dtm_debug_file)
        df_to_las(filled_df,dtm_header,dtm_debug_file)






In [ ]:
filled_df.columns

In [ ]:
dtm_file_path

In [ ]:
df['classification'].max()

In [ ]:
ground_df = df.query("classification==2")
non_ground_df = df.query("classification!=2")

In [ ]:
non_ground_df

In [ ]:
ground_df

In [ ]:
df_nan = df[['X',"Y","Z"]]
df_nan.loc[df['classification'] != 2, 'Z'] = np.nan    
imputer = SimpleImputer()
imputed_df = imputer.fit_transform(np.array(df_nan))

In [ ]:
imputed_df

In [ ]:

tree_top_points = np.array(non_ground_df[["X","Y","Z"]])
distances,indices = knn.kneighbors(tree_top_points)
def get_mean_z(row):
    nearest_ground_vectors = ground_points[index1] 
    mean_z = np.mean(nearest_ground_vectors[:,2],dtype='int32')
    return mean_z

z_points = [get_mean_z(x) for x in indices]


In [ ]:
tree_top_points[:,2] = z_points

In [ ]:
df[df['classification'] !=2] = z_points

In [ ]:
list(indices)